# Practical 12 — Metropolis Algorithm (Part A)

| Field | Details |
| --- | --- |
| **Name** | Md Ayan Alam |
| **Roll Number** | GF202342645 |
| **Course** | Statistical Foundation of Data Science |
| **Institution** | Shoolini University |



## Overview and Learning Objectives

- Implement the Metropolis algorithm to draw samples from a target distribution.
- Visualize trace plots, acceptance rates, and posterior approximations.
- Work incrementally to stay within length limits. Part B (Deterministic Model) will be appended later.

## 1. Environment Setup in VS Code

- Ensure the `.venv` at repo root is activated before running any cells.
- Kernel: Python 3.x (same as earlier practicals).
- VS Code Jupyter support: already enabled in this workspace.

In [ ]:
import sys
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")
print("Jupyter support detected. Proceeding with Practical 12 - Part A.")

## 2. Create Initial Notebook Structure

Add core imports, visualization defaults, and random seed (set to 42 for reproducibility).

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import warnings
warnings.filterwarnings("ignore")

# Visualization defaults
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 5)

# Reproducibility
np.random.seed(42)

print("Imports loaded. Seed set to 42.")

## 3. Add Data Loading Cell

For Part A (Metropolis), we'll use a simple synthetic target distribution (standard normal) to demonstrate sampling. We also keep a tiny synthetic dataset for comparison and summaries.

In [ ]:
# Synthetic reference dataset (not used by the sampler, but helpful for summaries)
synthetic_data = np.random.normal(loc=0.0, scale=1.0, size=500)

summary = pd.Series(synthetic_data).describe()
print("Synthetic reference dataset (N=500) summary:\n")
print(summary)

# Quick sanity checks
print("\nNaNs present:", np.isnan(synthetic_data).any())
print("Mean ~0?", np.isclose(np.mean(synthetic_data), 0, atol=0.15))

## 4. Implement Core Computation Cell — Metropolis Algorithm (Part A)

Target distribution: standard normal \(\mathcal{N}(0, 1)\). Proposal: symmetric normal random walk.

Steps:
1. Define unnormalized log-pdf for the target (log of standard normal up to constant).
2. Implement Metropolis sampler with burn-in and thinning options.
3. Run the chain and collect samples plus acceptance rate.

In [ ]:
def log_target_standard_normal(x: float) -> float:
    """Unnormalized log-pdf of standard normal (constant term omitted)."""
    return -0.5 * x ** 2


def metropolis_sampler(
    log_target,
    initial_value: float,
    proposal_std: float = 1.0,
    n_samples: int = 20000,
    burn_in: int = 2000,
    thin: int = 1,
):
    """
    Basic Metropolis random-walk sampler for a 1D target distribution.

    Parameters
    ----------
    log_target : callable
        Function returning log density up to a constant.
    initial_value : float
        Starting point for the chain.
    proposal_std : float, optional
        Standard deviation of the symmetric normal proposal.
    n_samples : int, optional
        Total iterations (including burn-in).
    burn_in : int, optional
        Number of initial iterations to discard.
    thin : int, optional
        Keep every `thin`-th sample after burn-in.

    Returns
    -------
    samples : np.ndarray
        Posterior samples after burn-in and thinning.
    acceptance_rate : float
        Overall acceptance rate across the full chain.
    trace : np.ndarray
        Full unthinned chain (useful for diagnostics).
    """
    chain = np.zeros(n_samples)
    chain[0] = initial_value
    accept_count = 0

    for i in range(1, n_samples):
        proposal = np.random.normal(chain[i - 1], proposal_std)
        log_alpha = log_target(proposal) - log_target(chain[i - 1])
        # Metropolis accept step
        if np.log(np.random.rand()) < log_alpha:
            chain[i] = proposal
            accept_count += 1
        else:
            chain[i] = chain[i - 1]

    acceptance_rate = accept_count / (n_samples - 1)

    # Burn-in and thinning
    kept = chain[burn_in:][::thin]
    return kept, acceptance_rate, chain


# Run sampler for Part A
samples, acc_rate, full_trace = metropolis_sampler(
    log_target=log_target_standard_normal,
    initial_value=0.5,
    proposal_std=0.8,
    n_samples=25000,
    burn_in=5000,
    thin=2,
)

print(f"Acceptance rate: {acc_rate:.3f}")
print(f"Kept samples: {len(samples)} (after burn-in/thinning)")
print(f"Sample mean: {np.mean(samples):.3f} (true mean 0)")
print(f"Sample std : {np.std(samples):.3f} (true std 1)")

## 5. Add Visualization Cell

Trace plot and posterior density vs. true standard normal pdf. Also compute basic diagnostics (autocorrelation at short lags).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trace plot
axes[0].plot(full_trace[:2000], color='#4ECDC4', linewidth=0.8)
axes[0].set_title('Trace (first 2,000 iterations)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Value')
axes[0].grid(True, alpha=0.3)

# Posterior histogram vs true PDF
x_grid = np.linspace(-4, 4, 400)
axes[1].hist(samples, bins=60, density=True, alpha=0.7, color='#FF6B6B', edgecolor='black', label='Posterior samples')
axes[1].plot(x_grid, stats.norm.pdf(x_grid, 0, 1), 'k--', linewidth=2, label='True N(0,1) pdf')
axes[1].set_title('Posterior vs. True Density', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Density')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Autocorrelation at short lags
for lag in [1, 5, 10, 20]:
    corr = np.corrcoef(full_trace[:-lag], full_trace[lag:])[0, 1]
    print(f"Lag {lag:>2} autocorrelation: {corr:.3f}")

## 6. Run and Inspect Outputs Incrementally

Execute cells in order: environment check → imports → data summary → Metropolis sampler → diagnostics. Confirm acceptance rate is in a reasonable band (typically 0.2–0.5 for random-walk proposals).

## 7. Refactor and Re-run Tests

Simple validation for Part A:
- Kolmogorov–Smirnov (KS) test comparing sampled distribution to \(\mathcal{N}(0,1)\).
- Re-run sampler with adjusted proposal if acceptance is too low/high.

Part B (Deterministic Model) will be appended after Part A is verified.

In [ ]:
# KS test against N(0,1)
ks_stat, ks_p = stats.kstest(samples, 'norm')
print(f"KS statistic: {ks_stat:.4f}, p-value: {ks_p:.4f}")

if ks_p < 0.05:
    print("Warning: KS test rejects N(0,1); consider tuning proposal_std or running longer.")
else:
    print("KS test does not reject N(0,1) — sampler looks reasonable.")

# Quick check for acceptance rate
if acc_rate < 0.15:
    print("Acceptance is low (<0.15); reduce proposal_std.")
elif acc_rate > 0.6:
    print("Acceptance is high (>0.6); increase proposal_std for better mixing.")
else:
    print("Acceptance in a good range (0.15–0.60).")